# 編集距離と単語の頻度で英語のスペル誤りを訂正する (Norvig 方式を自分で実装)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/B_implementations/spelling_correction_scratch.ipynb)

NLP クックブック B-21 ／ 仕組みを理解するための実装

英語のつづりの誤りを見つけて訂正したい。編集距離と単語の頻度による古典的な方法を作り、文脈を使う方法と比べる (ライブラリで手軽に直すだけなら、文法誤り訂正と言語教育の章の「英語のスペルミスをライブラリで直す (pyspellchecker)」)。

In [1]:
!pip install -q symspellpy
!wget -q https://norvig.com/big.txt https://norvig.com/spell-testset1.txt

In [2]:
import re
from collections import Counter

WORDS = Counter(re.findall(r"[a-z]+", open("big.txt", encoding="utf-8").read().lower()))     # 約 100 万語の英文から単語の頻度を数える
TOTAL = sum(WORDS.values())
LETTERS = "abcdefghijklmnopqrstuvwxyz"

def edits1(word):
    """編集距離 1 の文字列をすべて作る (削除・隣接文字の入れ替え・置換・挿入)"""
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    deletes = [a + b[1:] for a, b in splits if b]
    transposes = [a + b[1] + b[0] + b[2:] for a, b in splits if len(b) > 1]
    replaces = [a + c + b[1:] for a, b in splits if b for c in LETTERS]
    inserts = [a + c + b for a, b in splits for c in LETTERS]
    return set(deletes + transposes + replaces + inserts)

def known(words):
    return {w for w in words if w in WORDS}

def candidates(word):
    """辞書にある語を、編集距離の小さい順に探す"""
    return known([word]) or known(edits1(word)) or known(e2 for e1 in edits1(word) for e2 in edits1(e1)) or {word}

def correct(word):
    return max(candidates(word), key=WORDS.get)

for w in ["speling", "korrectud", "bycycle", "inconvient", "recieve", "definately", "their", "teh"]:
    print(f"{w} → {correct(w)}  (候補: {sorted(candidates(w), key=WORDS.get, reverse=True)[:5]})")

speling → spelling  (候補: ['spelling'])
korrectud → corrected  (候補: ['corrected'])
bycycle → bicycle  (候補: ['bicycle'])
inconvient → inconvenient  (候補: ['inconvenient', 'convient'])
recieve → receive  (候補: ['receive', 'relieve'])
definately → definitely  (候補: ['definitely'])
their → their  (候補: ['their'])
teh → the  (候補: ['the', 'ten', 'tea', 'eh', 'th'])


In [3]:
# 誤りと正解の組のテストセットで正解率を測る
tests = [(wrong, right) for line in open("spell-testset1.txt") for right, wrongs in [line.strip().split(":")] for wrong in wrongs.split()]

def accuracy(corrector, tests):
    result = [corrector(wrong) == right for wrong, right in tests]
    return sum(result) / len(result)

print(len(tests), "組 / 正解率:", round(accuracy(correct, tests), 3), "/ 正解の語が辞書にない割合:", round(sum(r not in WORDS for _, r in tests) / len(tests), 3))
errors = [(w, correct(w), r) for w, r in tests if correct(w) != r][:8]
print("誤りの例 (入力, 出力, 正解):", errors)

270 組 / 正解率: 0.748 / 正解の語が辞書にない割合: 0.056
誤りの例 (入力, 出力, 正解): [('contende', 'contended', 'contented'), ('contended', 'contended', 'contented'), ('proplen', 'people', 'problem'), ('guic', 'guns', 'juice'), ('juce', 'june', 'juice'), ('jucie', 'julie', 'juice'), ('juise', 'guise', 'juice'), ('juse', 'just', 'juice')]


In [4]:
# 高速なライブラリ: SymSpell (削除だけで候補を引く索引を作っておくので、非常に速い)
import time
from symspellpy import SymSpell, Verbosity

sym = SymSpell(max_dictionary_edit_distance=2)
for word, count in WORDS.items():
    sym.create_dictionary_entry(word, count)

def correct_symspell(word):
    hits = sym.lookup(word, Verbosity.TOP, max_edit_distance=2)
    return hits[0].term if hits else word

for name, f in [("自作", correct), ("SymSpell", correct_symspell)]:
    t = time.perf_counter(); acc = accuracy(f, tests)
    print(f"{name}: 正解率 {acc:.3f} / {len(tests) / (time.perf_counter() - t):.0f} 語/秒")

自作: 正解率 0.748 / 70 語/秒
SymSpell: 正解率 0.748 / 19054 語/秒


In [5]:
# 文脈を使う: 候補のうち、マスク言語モデルが文脈に合うと判断したものを選ぶ (辞書にある語への打ち間違いにも対応できる)
import math
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
mlm = AutoModelForMaskedLM.from_pretrained("distilbert-base-uncased").eval()

@torch.no_grad()
def correct_in_context(sentence, position, penalty=2.0):
    words = sentence.split()
    word = words[position].lower()
    near = known(edits1(word))
    far = known(e2 for e1 in edits1(word) for e2 in edits1(e1)) - near      # peace → piece のように、実在語どうしの取り違えは編集距離 2 のことが多い
    distance = {word: 0, **{c: 1 for c in near - {word}}, **{c: 2 for c in far - {word}}}
    enc = tokenizer(" ".join(words[:position] + [tokenizer.mask_token] + words[position + 1:]), return_tensors="pt")
    pos = (enc["input_ids"][0] == tokenizer.mask_token_id).nonzero()[0, 0]
    logp = mlm(**enc).logits[0, pos].log_softmax(-1)
    def score(c):
        ids = tokenizer(c, add_special_tokens=False)["input_ids"]
        context = float(logp[ids[0]]) if len(ids) == 1 else -20.0               # 1 トークンで表せない語は文脈のスコアを使わない
        return context - penalty * distance[c]                                   # 元の語から離れるほどペナルティ
    return max(distance, key=score)

for sentence, position in [("I want a peace of cake .", 3), ("He will except the offer .", 2), ("She lives in a quite village .", 4),
                           ("The whether is nice today .", 1), ("Their is a cat on the table .", 0), ("I need to buy some bred .", 5),
                           ("I like to read books .", 4), ("The cat sat on the mat .", 1)]:
    print(sentence, "→", sentence.split()[position], "⇒", correct_in_context(sentence, position), "(単語だけで訂正:", correct(sentence.split()[position].lower()) + ")")

I want a peace of cake . → peace ⇒ piece (単語だけで訂正: peace)
He will except the offer . → except ⇒ accept (単語だけで訂正: except)
She lives in a quite village . → quite ⇒ quiet (単語だけで訂正: quite)
The whether is nice today . → whether ⇒ weather (単語だけで訂正: whether)
Their is a cat on the table . → Their ⇒ there (単語だけで訂正: their)
I need to buy some bred . → bred ⇒ bread (単語だけで訂正: bred)
I like to read books . → books ⇒ books (単語だけで訂正: books)
The cat sat on the mat . → cat ⇒ cat (単語だけで訂正: cat)


- 古典的なスペル訂正は「編集距離が近い語のうち、最も頻度の高い語を選ぶ」だけで、かなりの誤りを直せる。Peter Norvig の有名な 20 数行の実装がこれにあたる。
- 弱点は 3 つ: 辞書にない語は正解にできない、頻度だけで選ぶので文脈に合わない語を選ぶことがある、そして実在する別の語になってしまった誤り (peace / piece) には気付けない。
- 文脈を使うと、実在する語への打ち間違い (real-word error) も直せる (peace → piece、whether → weather)。こうした取り違えは編集距離 2 のことが多いので、候補を距離 2 まで広げる。正しい語まで書き換えてしまう危険が増えるので、元の語から離れるほどペナルティを付け、正しい文 (books、cat) が変わらないことも確かめる。
- 学習者のスペル誤りは、打ち間違いとは傾向が違う (発音に引きずられる、母語の影響を受ける)。編集操作ごとの起こりやすさを学習者のデータから推定すると、精度が上がる。
- 日本語には分かち書きがなく、スペル誤りにあたるのは誤変換や脱字。日本語入力の章と、文法誤り検出のレシピを参照。
- 実務でライブラリを使って同じことをする版は、文法誤り訂正と言語教育の章の「symspellpy とマスク言語モデルで英語のスペル誤りを文脈に合わせて直す」。